# Data Cleaning & Feature Engineering Pipeline
## Thesis: Skill-based Career Path Modeling and Recommendation

This notebook handles the ETL (Extract, Transform, Load) process for the National Exam datasets (2017-2025). It addresses schema drift, standardizes categorical variables, normalizes scores, and engineers new skill-based features suitable for Machine Learning models.

In [1]:
import pandas as pd
import numpy as np
import glob
import os

# Define the path to the data directory
data_dir = '../DATA/'
all_files = glob.glob(os.path.join(data_dir, "National exam_dataset_S3_*.csv"))
print(f"Found {len(all_files)} files to process.")

Found 8 files to process.


### 1. Schema Standardization (Handling Schema Drift)
We create a comprehensive mapping dictionary to unify all variations of column names into a standard format.

In [2]:
column_mapping = {
    # Biology
    'grade value biology': 'Biology_Score',
    'biology grade value': 'Biology_Score',
    'biology and health sciences raw marks': 'Biology_Score',
    'biology_grade_letter': 'Biology_Grade_Letter',
    
    # Chemistry
    'grade value chemistry': 'Chemistry_Score',
    'chemistry grade value': 'Chemistry_Score',
    'chemistry raw marks': 'Chemistry_Score',
    'chemistry_grade_letter': 'Chemistry_Grade_Letter',
    
    # Mathematics
    'grade value maths': 'Math_Score',
    'mathematics grade value': 'Math_Score',
    'mathematics raw marks': 'Math_Score',
    'mathematics_grade_letter': 'Math_Grade_Letter',
    
    # Physics
    'grade value physics': 'Physics_Score',
    'physics grade value': 'Physics_Score',
    'physics raw marks': 'Physics_Score',
    'physics_grade_letter': 'Physics_Grade_Letter',
    
    # Identifiers & Metadata
    'school code': 'School_Code',
    'student number': 'Student_Number',
    'option': 'Option',
    'school year': 'School_Year',
    'candidate id': 'Candidate_ID',
    'combination': 'Combination'
}

df_list = []
for file in all_files:
    try:
        # Read each CSV, treating specific columns as strings to preserve leading zeros
        df = pd.read_csv(file, dtype={'school code': str, 'student number': str})
        # Lowercase and strip column names before mapping
        df.columns = df.columns.str.strip().str.lower()
        # Rename columns using our dictionary
        df.rename(columns=column_mapping, inplace=True)
        df_list.append(df)
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Concatenate all files into a single dataframe
master_df = pd.concat(df_list, ignore_index=True)
print(f"Master dataset shape: {master_df.shape}")

/var/folders/74/yr6kbjmd3d92m1cgxsmjqryh0000gn/T/ipykernel_17464/706392352.py:39: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, dtype={'school code': str, 'student number': str})


Master dataset shape: (983958, 16)


### 2. Categorical Value Cleaning
Standardizing the 'Option' column to merge inconsistent data entry (e.g., '0LC' -> 'OLC').

In [3]:
if 'Option' in master_df.columns:
    # Convert to uppercase and strip whitespace
    master_df['Option'] = master_df['Option'].astype(str).str.strip().str.upper()
    # Handle specific known typos
    master_df['Option'] = master_df['Option'].replace({'0LC': 'OLC'})
    
    print("Top Options after cleaning:")
    print(master_df['Option'].value_counts().head(5))

Top Options after cleaning:
Option
NAN     673385
OLC     277825
TVET     32747
PRI          1
Name: count, dtype: int64


### 3. Score Normalization (Critical for Machine Learning)
Some years use "raw marks" (0-100), while others use "grade values" (1-9). 
To use these as generic "skills", we must normalize them to a common scale (e.g., Min-Max scaling 0 to 1). 
This cell applies a robust year-by-year or global Min-Max scaling approach so algorithms weigh all skills equally.

In [4]:
subjects = ['Math_Score', 'Physics_Score', 'Chemistry_Score', 'Biology_Score']

# Normalization loop
for sub in subjects:
    if sub in master_df.columns:
        # Convert to numeric, forcing errors to NaN
        master_df[sub] = pd.to_numeric(master_df[sub], errors='coerce')
        
        # For demonstration, we apply a global Min-Max scaler across the column.
        # Note for Thesis: A more rigorous approach processes this *before* concat based on each year's schema.
        min_val = master_df[sub].min()
        max_val = master_df[sub].max()
        
        if pd.notna(min_val) and pd.notna(max_val) and max_val > min_val:
             master_df[f'{sub}_Normalized'] = (master_df[sub] - min_val) / (max_val - min_val)
        else:
             master_df[f'{sub}_Normalized'] = np.nan

print("Normalized Summary:")
print(master_df[[f'{s}_Normalized' for s in subjects if s in master_df.columns]].describe())

Normalized Summary:
       Math_Score_Normalized  Physics_Score_Normalized  \
count          837631.000000             837843.000000   
mean                0.128900                  0.117918   
std                 0.187177                  0.156666   
min                 0.000000                  0.000000   
25%                 0.030000                  0.030000   
50%                 0.060000                  0.070000   
75%                 0.090000                  0.090000   
max                 1.000000                  1.000000   

       Chemistry_Score_Normalized  Biology_Score_Normalized  
count               837826.000000             837824.000000  
mean                     0.144893                  0.129226  
std                      0.204858                  0.184907  
min                      0.000000                  0.000000  
25%                      0.030000                  0.030000  
50%                      0.080000                  0.060000  
75%                    

### 4. Feature Engineering: Skill Composites
We create broader skill categories by combining normalized subject scores to build the 'Skill Profile'.

In [5]:
# 1. Quantitative & Analytical Skill (Math + Physics)
if 'Math_Score_Normalized' in master_df.columns and 'Physics_Score_Normalized' in master_df.columns:
    master_df['Quantitative_Skill'] = master_df[['Math_Score_Normalized', 'Physics_Score_Normalized']].mean(axis=1)

# 2. Scientific & Research Skill (Biology + Chemistry)
if 'Biology_Score_Normalized' in master_df.columns and 'Chemistry_Score_Normalized' in master_df.columns:
    master_df['Scientific_Skill'] = master_df[['Biology_Score_Normalized', 'Chemistry_Score_Normalized']].mean(axis=1)

print("Sample of Engineered Skill Profiles:")
print(master_df[['Student_Number', 'Option', 'Quantitative_Skill', 'Scientific_Skill']].head())

Sample of Engineered Skill Profiles:
  Student_Number Option  Quantitative_Skill  Scientific_Skill
0            NaN    NAN                 NaN               NaN
1            NaN    NAN                 NaN               NaN
2            NaN    NAN                 NaN               NaN
3            NaN    NAN                 NaN               NaN
4            NaN    NAN                 NaN               NaN


### 5. Final Export
Save the cleaned and feature-engineered dataset for use in Machine Learning models (like K-Means Clustering or Recommendation engines).

In [10]:
output_file = '../DATA/cleaned_skill_profiles.csv'

# Drop rows where all critical skill scores are NaN to keep the dataset clean
final_df = master_df.dropna(subset=['Quantitative_Skill', 'Scientific_Skill'], how='all')

# Uncomment the line below to execute the save command
# final_df.to_csv(output_file, index=False)

print(f"Data ready for ML Modeling. Final Shape: {final_df.shape}")

Data ready for ML Modeling. Final Shape: (838051, 22)
